<a href="https://colab.research.google.com/github/francois-gonon/POC2PROD_intro/blob/main/notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive; drive.mount('/content/drive')
base_path = 'drive/MyDrive/POC2PROD/' # change to 'drive/MyDrive/POC2PROD/' for Colab or './' if running locally

Mounted at /content/drive


In [2]:
import pandas as pd
import torch
import re
import os
import json
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
import matplotlib.pyplot as plt

In [3]:
# Load data
df = pd.read_csv(os.path.join(base_path, 'stackoverflow_posts.csv'))
if 'tag_id' not in df.columns:
    tag_map = {t: i for i, t in enumerate(sorted(df['tag_name'].dropna().unique()))}
    df['tag_id'] = df['tag_name'].map(tag_map)
df = df.dropna(subset=['title', 'tag_id'])
df.head()

,post_id,tag_name,tag_id,tag_position,title
0,1987528,php,5,0,Is it possible to execute the procedure of a f...
1,1987531,ruby-on-rails,4984,0,ruby on rails: how to change BG color of optio...
2,1987531,list,5608,1,ruby on rails: how to change BG color of optio...
3,1987531,select,1151,2,ruby on rails: how to change BG color of optio...
4,1987538,c#,9,0,How to read tags out of m4a files in .NET?


In [4]:
# Preprocess text
def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'[^a-z\s]', '', text)
    return text
df['clean_title'] = df['title'].astype(str).apply(preprocess_text)
df.head()

,post_id,tag_name,tag_id,tag_position,title,clean_title
0,1987528,php,5,0,Is it possible to execute the procedure of a f...,is it possible to execute the procedure of a f...
1,1987531,ruby-on-rails,4984,0,ruby on rails: how to change BG color of optio...,ruby on rails how to change bg color of option...
2,1987531,list,5608,1,ruby on rails: how to change BG color of optio...,ruby on rails how to change bg color of option...
3,1987531,select,1151,2,ruby on rails: how to change BG color of optio...,ruby on rails how to change bg color of option...
4,1987538,c#,9,0,How to read tags out of m4a files in .NET?,how to read tags out of ma files in net


In [5]:
# Remove rare labels globally first
label_counts = df['tag_id'].value_counts()
rare_labels = label_counts[label_counts < 6].index
if len(rare_labels) > 0:
    df = df[~df['tag_id'].isin(rare_labels)]

# Split data for stage 1: only tag_position = 0
df_stage1 = df[df['tag_position'] == 0].copy()
label_counts_stage1 = df_stage1['tag_id'].value_counts()
rare_labels_stage1 = label_counts_stage1[label_counts_stage1 < 6].index
if len(rare_labels_stage1) > 0:
    df_stage1 = df_stage1[~df_stage1['tag_id'].isin(rare_labels_stage1)]
train_df_stage1, test_df_stage1 = train_test_split(df_stage1, test_size=0.2, stratify=df_stage1['tag_id'], random_state=42)

# Split data for stage 2: full dataset
train_df_full, test_df_full = train_test_split(df, test_size=0.2, stratify=df['tag_id'], random_state=42)

In [6]:
# Load tokenizer and model
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
label_dict = {label: i for i, label in enumerate(train_df_stage1['tag_id'].unique())}
num_labels = len(label_dict)
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=num_labels)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [7]:
# Create dataset class
class Dataset(torch.utils.data.Dataset):
    def __init__(self, df, tokenizer, label_dict):
        self.df = df
        self.tokenizer = tokenizer
        self.label_dict = label_dict
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        text = self.df.iloc[idx]['clean_title']
        label = self.label_dict[self.df.iloc[idx]['tag_id']]
        encoding = self.tokenizer(text, truncation=True, padding='max_length', max_length=128, return_tensors='pt')
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

train_dataset_stage1 = Dataset(train_df_stage1, tokenizer, label_dict)
test_dataset_stage1 = Dataset(test_df_stage1, tokenizer, label_dict)

In [8]:
# Stage 1: Train on tag_position = 0 subset
training_args = TrainingArguments(
    output_dir=os.path.join(base_path, 'results_stage1'),
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    logging_dir=os.path.join(base_path, 'logs_stage1'),
)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset_stage1,
    eval_dataset=test_dataset_stage1,
)
trainer.train()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: francoisg (francoisg-epf-engineering-school) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Step,Training Loss
500,2.838900


TrainOutput(global_step=618, training_loss=2.6865898798970345, metrics={'train_runtime': 454.56, 'train_samples_per_second': 21.753, 'train_steps_per_second': 1.36, 'total_flos': 650860191424512.0, 'train_loss': 2.6865898798970345, 'epoch': 3.0})

In [9]:
# Evaluate stage 1
predictions = trainer.predict(test_dataset_stage1)
pred_labels = [list(label_dict.keys())[p] for p in predictions.predictions.argmax(axis=1)]
true_labels = test_df_stage1['tag_id'].tolist()
acc = accuracy_score(true_labels, pred_labels)
f1 = f1_score(true_labels, pred_labels, average='weighted')
print(f'Stage 1 - Test acc: {acc:.4f}, f1: {f1:.4f}')
print(classification_report(true_labels, pred_labels))

# Save stage 1 model
model.save_pretrained(os.path.join(base_path, 'model_stage1'))
tokenizer.save_pretrained(os.path.join(base_path, 'model_stage1'))
with open(os.path.join(base_path, 'model_stage1', 'label_dict.json'), 'w') as f:
    json.dump({str(k): v for k, v in label_dict.items()}, f)

Stage 1 - Test acc: 0.4551, f1: 0.3875
              precision    recall  f1-score   support

           1       0.17      0.04      0.06        25
           2       0.53      0.53      0.53        17
           3       0.52      0.56      0.54        45
           4       0.00      0.00      0.00         8
           5       0.58      0.65      0.62        92
           8       0.00      0.00      0.00        15
           9       0.32      0.67      0.44       101
          10       0.12      0.05      0.08        37
          12       0.00      0.00      0.00         7
          16       0.55      0.66      0.60        44
          17       0.45      0.75      0.56        71
          18       0.00      0.00      0.00         4
          19       0.00      0.00      0.00         5
          21       0.00      0.00      0.00        10
          22       0.37      0.55      0.44        20
          28       0.00      0.00      0.00         2
          30       0.00      0.00      0.0

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [10]:
# Update label dict for full dataset
label_dict_full = {label: i for i, label in enumerate(train_df_full['tag_id'].unique())}
num_labels_full = len(label_dict_full)

# Load new model for stage 2
model_stage2 = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=num_labels_full)

train_dataset_full = Dataset(train_df_full, tokenizer, label_dict_full)
test_dataset_full = Dataset(test_df_full, tokenizer, label_dict_full)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [11]:
# Stage 2: Train on full dataset
training_args_full = TrainingArguments(
    output_dir=os.path.join(base_path, 'results_stage2'),
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    logging_dir=os.path.join(base_path, 'logs_stage2'),
)
trainer_full = Trainer(
    model=model_stage2,
    args=training_args_full,
    train_dataset=train_dataset_full,
    eval_dataset=test_dataset_full,
)
trainer_full.train()

Step,Training Loss
500,4.957300
1000,4.265000


TrainOutput(global_step=1353, training_loss=4.443111715718542, metrics={'train_runtime': 481.9377, 'train_samples_per_second': 44.863, 'train_steps_per_second': 2.807, 'total_flos': 1426433172132096.0, 'train_loss': 4.443111715718542, 'epoch': 3.0})

In [12]:
# Evaluate stage 2
predictions_full = trainer_full.predict(test_dataset_full)
pred_labels_full = [list(label_dict_full.keys())[p] for p in predictions_full.predictions.argmax(axis=1)]
true_labels_full = test_df_full['tag_id'].tolist()
acc_full = accuracy_score(true_labels_full, pred_labels_full)
f1_full = f1_score(true_labels_full, pred_labels_full, average='weighted')
print(f'Stage 2 - Test acc: {acc_full:.4f}, f1: {f1_full:.4f}')
print(classification_report(true_labels_full, pred_labels_full))

# Save stage 2 model
model_stage2.save_pretrained(os.path.join(base_path, 'model_stage2'))
tokenizer.save_pretrained(os.path.join(base_path, 'model_stage2'))
with open(os.path.join(base_path, 'model_stage2', 'label_dict.json'), 'w') as f:
    json.dump({str(k): v for k, v in label_dict_full.items()}, f)

Stage 2 - Test acc: 0.2309, f1: 0.1475
              precision    recall  f1-score   support

           1       0.11      0.09      0.09        47
           2       0.18      0.27      0.22        33
           3       0.22      0.39      0.28        54
           4       0.00      0.00      0.00        20
           5       0.28      0.55      0.37        94
           8       0.00      0.00      0.00        21
           9       0.15      0.60      0.24       101
          10       0.08      0.16      0.11        38
          12       0.24      0.36      0.29        14
          16       0.27      0.66      0.39        47
          17       0.25      0.65      0.36        72
          18       0.00      0.00      0.00        12
          19       0.00      0.00      0.00        12
          21       0.30      0.45      0.36        31
          22       0.17      0.56      0.27        27
          23       0.00      0.00      0.00         4
          27       0.00      0.00      0.0

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
